## ICU Charting and Staff Data

This notebook explores the five MIMIC demo tables assigned to the ICU charting and staff data area.

The first step is to check the size, structure and main identifiers in each table before deciding what needs to be cleaned for the Bronze and Silver layers.

In [0]:
SOURCE_PATH = "s3://mimic-iii-demo-bucket/"

def load_mimic_csv(file_name):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{SOURCE_PATH}{file_name}")
    )

chartevents = load_mimic_csv("CHARTEVENTS.csv")
d_items = load_mimic_csv("D_ITEMS.csv")
datetimeevents = load_mimic_csv("DATETIMEEVENTS.csv")
caregivers = load_mimic_csv("CAREGIVERS.csv")
callout = load_mimic_csv("CALLOUT.csv")

datasets = {
    "CHARTEVENTS": chartevents,
    "D_ITEMS": d_items,
    "DATETIMEEVENTS": datetimeevents,
    "CAREGIVERS": caregivers,
    "CALLOUT": callout
}

for name, df in datasets.items():
    print(f"{name}: {df.count():,} rows, {len(df.columns)} columns")

### Initial Findings

- `CHARTEVENTS` is by far the largest table with over 758,000 rows, so this will probably be the main table where performance and filtering matter.
- `DATETIMEEVENTS` is much smaller but still looks like another event-level table that will likely link back to patients, admissions and ICU stays.
- `D_ITEMS` and `CAREGIVERS` look more like supporting reference tables that can be used to explain item and staff IDs found in the event data.
- `CALLOUT` is very small with only 77 rows, although it has a relatively large number of columns.

In [0]:
for name, df in datasets.items():
    print(f"\n{name}")
    df.printSchema()

### Schema Findings

- `CHARTEVENTS` and `DATETIMEEVENTS` have a very similar event structure and both link records back to a patient, hospital admission and ICU stay through `subject_id`, `hadm_id` and `icustay_id`.
- Both event tables also contain `itemid`, which should link to `D_ITEMS` to explain what each recorded measurement or event represents.
- `cgid` appears in both event tables and should link to `CAREGIVERS`, giving some context around the staff member associated with the recorded event.
- `CALLOUT` is structured differently and appears to relate more to movement or discharge requests between hospital wards. It links at the patient and admission level through `subject_id` and `hadm_id`, but does not contain `icustay_id`.
- The timestamp columns have already been inferred correctly, which should make the later Silver cleaning easier.

In [0]:
key_checks = {
    "CHARTEVENTS": ["row_id", "subject_id", "hadm_id", "icustay_id", "itemid", "cgid"],
    "D_ITEMS": ["row_id", "itemid"],
    "DATETIMEEVENTS": ["row_id", "subject_id", "hadm_id", "icustay_id", "itemid", "cgid"],
    "CAREGIVERS": ["row_id", "cgid"],
    "CALLOUT": ["row_id", "subject_id", "hadm_id"]
}

for name, keys in key_checks.items():
    df = datasets[name]
    print(f"\n{name}")

    for key in keys:
        total = df.count()
        unique = df.select(key).distinct().count()
        nulls = df.filter(df[key].isNull()).count()

        print(f"{key}: {unique:,} unique | {nulls:,} nulls")

### Key Findings

- `row_id` is unique with no missing values across all five tables, so it looks like a reliable row-level identifier.
- `D_ITEMS.itemid` is fully unique, confirming it works as a lookup table for the `itemid` values used in the event tables.
- `CAREGIVERS.cgid` is also fully unique, so it can be used as the lookup table for caregiver IDs.
- `CHARTEVENTS` contains 1,679 different item types and 592 caregivers, showing how much more detailed this table is compared with the others.
- `icustay_id` is missing for a small number of rows in both event tables. This may be valid for events recorded outside a specific ICU stay, so it should be investigated rather than automatically removed.
- `CALLOUT` links at the patient and hospital admission level rather than directly to an ICU stay.

In [0]:
# Check lookup coverage for event table keys
checks = [
    ("CHARTEVENTS itemid", chartevents, "itemid", d_items, "itemid"),
    ("CHARTEVENTS cgid", chartevents, "cgid", caregivers, "cgid"),
    ("DATETIMEEVENTS itemid", datetimeevents, "itemid", d_items, "itemid"),
    ("DATETIMEEVENTS cgid", datetimeevents, "cgid", caregivers, "cgid")
]

for name, left_df, left_key, right_df, right_key in checks:
    unmatched = (
        left_df
        .select(left_key)
        .distinct()
        .join(
            right_df.select(right_key).distinct(),
            left_df[left_key] == right_df[right_key],
            "left_anti"
        )
        .count()
    )

    print(f"{name}: {unmatched:,} unmatched keys")

### Relationship Findings

- All `itemid` values in `CHARTEVENTS` and `DATETIMEEVENTS` successfully match to `D_ITEMS`.
- All `cgid` values in both event tables also match to `CAREGIVERS`.
- This confirms the main lookup relationships are complete, so these tables should join cleanly in the Silver layer.
- The main data-quality questions now move away from broken relationships and more towards missing values, inconsistent fields and what columns are actually useful.

In [0]:
from pyspark.sql import functions as F

for name, df in datasets.items():
    total = df.count()

    null_summary = [
        (
            column,
            df.filter(F.col(column).isNull()).count()
        )
        for column in df.columns
    ]

    null_summary = [
        (column, nulls, round((nulls / total) * 100, 2))
        for column, nulls in null_summary
        if nulls > 0
    ]

    print(f"\n{name}")

    if not null_summary:
        print("No missing values")
    else:
        for column, nulls, pct in sorted(null_summary, key=lambda x: x[2], reverse=True):
            print(f"{column}: {nulls:,} nulls ({pct}%)")

### Missing Value Findings

- `CHARTEVENTS` has several heavily incomplete fields, particularly `resultstatus`, `valueuom` and `valuenum`.
- The missing `valuenum` values may be expected because some charted values are text rather than numeric, so this should be checked before cleaning.
- `D_ITEMS.conceptid` is completely empty in the demo data and is likely not useful for the Silver layer.
- `DATETIMEEVENTS.resultstatus` is also completely empty, while most of the important identifiers and timestamps are well populated.
- `CAREGIVERS` is mostly complete, with missing values mainly limited to the optional `description` field.
- `CALLOUT` has several fields with high missingness, particularly reservation and care unit fields, which may only apply to certain callout records.
- Overall, the main identifiers are very complete. Most missing values appear to be in optional descriptive or event-specific fields rather than the core relationships.

In [0]:
# Check values in some of the main nullable fields
fields_to_check = {
    "CHARTEVENTS": ["warning", "error", "resultstatus", "stopped"],
    "D_ITEMS": ["category", "param_type", "unitname"],
    "DATETIMEEVENTS": ["warning", "error", "stopped"],
    "CALLOUT": ["submit_careunit", "callout_status", "callout_outcome"]
}

for name, columns in fields_to_check.items():
    print(f"\n{name}")

    for column in columns:
        print(f"\n{column}")
        datasets[name].groupBy(column).count().orderBy(F.desc("count")).show(10, truncate=False)

### Field Pattern Findings

- In `CHARTEVENTS`, `warning` and `error` are populated for roughly half the records, while `resultstatus` and `stopped` are populated more heavily in the other half. This looks more like a difference between groups of records than general missing data.
- `DATETIMEEVENTS` shows a similar pattern, with `warning` and `error` populated together and `stopped` populated for a different group of records.
- `D_ITEMS` contains useful metadata such as `category`, `param_type` and `unitname`, but these fields are only available for some item types. They should be kept for now rather than removed based only on null percentage.
- `CALLOUT.callout_status` has no variation in the demo data, while `callout_outcome` mainly separates discharged and cancelled requests.
- Overall, the Silver cleaning should preserve meaningful nulls rather than treating all missing values as data-quality problems.

In [0]:
# Check source system distribution
print("D_ITEMS")
d_items.groupBy("dbsource").count().orderBy(F.desc("count")).show(truncate=False)

print("CHARTEVENTS")
(
    chartevents
    .join(d_items.select("itemid", "dbsource"), on="itemid", how="left")
    .groupBy("dbsource")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

print("DATETIMEEVENTS")
(
    datetimeevents
    .join(d_items.select("itemid", "dbsource"), on="itemid", how="left")
    .groupBy("dbsource")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

### Source System Findings

- The event data is split between `CareVue` and `MetaVision`, which appear to record some fields differently.
- In `CHARTEVENTS`, all 376,076 `CareVue` records line up exactly with the records where `warning` and `error` are missing.
- `DATETIMEEVENTS` shows the same pattern, with all 4,637 `CareVue` records missing `warning` and `error`, while the 10,914 `MetaVision` records contain them.
- This means these nulls should not be treated as errors or automatically filled during Silver cleaning.
- `dbsource` will be useful to keep in the Silver layer because it explains differences in how records were captured.

In [0]:
# Compare nullable fields by source system
chartevents_source = chartevents.join(
    d_items.select("itemid", "dbsource"),
    on="itemid",
    how="left"
)

(
    chartevents_source
    .groupBy("dbsource")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("warning").isNull().cast("int")).alias("warning_nulls"),
        F.sum(F.col("error").isNull().cast("int")).alias("error_nulls"),
        F.sum(F.col("resultstatus").isNull().cast("int")).alias("resultstatus_nulls"),
        F.sum(F.col("stopped").isNull().cast("int")).alias("stopped_nulls")
    )
    .show(truncate=False)
)

### Source-Specific Field Findings

- `MetaVision` records always contain `warning` and `error`, but do not use `resultstatus` or `stopped`.
- `CareVue` shows the opposite pattern, with `warning` and `error` completely missing while `stopped` is populated for almost every record.
- `resultstatus` is only used for a small portion of the `CareVue` data.
- These fields should therefore be kept as source-specific attributes in Silver rather than filled with replacement values.
- This also confirms that `dbsource` should be carried into the cleaned event tables so these differences remain clear.

In [0]:
# Check how chart values are stored by source
(
    chartevents_source
    .groupBy("dbsource")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("value").isNull().cast("int")).alias("value_nulls"),
        F.sum(F.col("valuenum").isNull().cast("int")).alias("valuenum_nulls"),
        F.sum(F.col("valueuom").isNull().cast("int")).alias("valueuom_nulls")
    )
    .show(truncate=False)
)

### Chart Value Findings

- Both source systems contain a mix of numeric and non-numeric charted values, which explains why `valuenum` is missing for a large number of records.
- `value` is much more complete than `valuenum`, especially in `MetaVision`, so it should remain the main raw representation of what was recorded.
- `valueuom` is also missing quite often, which is likely because many charted observations are not measurements that require a unit.
- This means we should not treat missing `valuenum` or `valueuom` as errors without first checking what type of item was recorded.

In [0]:
# Compare value storage by item type
chartevents_items = chartevents_source.join(
    d_items.select("itemid", "param_type"),
    on="itemid",
    how="left"
)

(
    chartevents_items
    .groupBy("param_type")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("value").isNull().cast("int")).alias("value_nulls"),
        F.sum(F.col("valuenum").isNull().cast("int")).alias("valuenum_nulls"),
        F.sum(F.col("valueuom").isNull().cast("int")).alias("valueuom_nulls")
    )
    .orderBy(F.desc("rows"))
    .show(truncate=False)
)

### Value Type Findings

- The missing `param_type` values line up with the `CareVue` records, so this metadata is mainly available for `MetaVision`.
- Numeric item types consistently use `valuenum`, while text items mostly rely on the original `value` field.
- `valueuom` is mainly relevant for numeric measurements and is naturally missing for text and checkbox fields.
- This confirms that `value`, `valuenum` and `valueuom` should all be kept in Silver rather than trying to force every record into the same format.
- Any validation of chart values will need to consider both the item type and source system.

In [0]:
# Check chart and storage time consistency
for name, df in {
    "CHARTEVENTS": chartevents,
    "DATETIMEEVENTS": datetimeevents
}.items():

    total = df.count()
    before_chart = df.filter(F.col("storetime") < F.col("charttime")).count()
    same_time = df.filter(F.col("storetime") == F.col("charttime")).count()
    after_chart = df.filter(F.col("storetime") > F.col("charttime")).count()

    print(f"\n{name}")
    print(f"Stored before chart time: {before_chart:,}")
    print(f"Stored at chart time:     {same_time:,}")
    print(f"Stored after chart time:  {after_chart:,}")

### Timestamp Findings

- Both event tables have complete `charttime` and `storetime` values.
- Most `CHARTEVENTS` records were stored after they were charted, although around 96,000 were stored earlier.
- `DATETIMEEVENTS` has a more even mix of records stored at or after the chart time.
- Records stored before `charttime` should not be removed automatically. We first need to check whether this behaviour is linked to the source system.

In [0]:
# Compare timestamp behaviour by source system
for name, df in {
    "CHARTEVENTS": chartevents,
    "DATETIMEEVENTS": datetimeevents
}.items():

    df_source = df.join(
        d_items.select("itemid", "dbsource"),
        on="itemid",
        how="left"
    )

    print(f"\n{name}")

    (
        df_source
        .groupBy("dbsource")
        .agg(
            F.count("*").alias("rows"),
            F.sum((F.col("storetime") < F.col("charttime")).cast("int")).alias("stored_before"),
            F.sum((F.col("storetime") == F.col("charttime")).cast("int")).alias("stored_same"),
            F.sum((F.col("storetime") > F.col("charttime")).cast("int")).alias("stored_after")
        )
        .show(truncate=False)
    )

### Exploration Summary

The initial profiling shows that the five datasets have clear relationships and generally reliable key fields.

`CHARTEVENTS` and `DATETIMEEVENTS` contain the main ICU event data and connect to `D_ITEMS` through `itemid` and `CAREGIVERS` through `cgid`. `CALLOUT` is separate from the charting data and links mainly through patient and admission identifiers.

Most key fields are complete and all tested lookup relationships matched successfully. A number of highly missing fields were found, although much of this is caused by differences between the CareVue and MetaVision source systems rather than poor data quality.

Based on this, the Bronze layer should preserve the source data with minimal changes. The Silver layer can then focus on useful cleaning, standardisation and retaining the source information needed to interpret these differences.